
Examples demonstrating step-wise outlier detection orchestration.

The StepwiseOutlierDetection class chains multiple outlier detection methods
sequentially. Each method operates on results from the previous one, progressively
filtering outliers. This example shows 6 methods with full parameter signatures:

1. Hampel filter — Median Absolute Deviation in sliding window
2. LocalSD — Local standard deviation from rolling median
3. Z-score global — Single threshold across all data
4. Z-score rolling — Adaptive threshold from rolling statistics
5. Z-score day/night — Separate thresholds for daytime/nighttime
6. Increments — Detect abrupt changes between consecutive values

See diive.pkgs.preprocessing.outlierdetection.stepwiseoutlierdetection for complete API.


In [ ]:
import matplotlib.pyplot as plt

import diive as dv


def example_stepwise_outlier_detection_synthetic_data():
    """Demonstrate stepwise detection with multiple methods on synthetic noisy data.

    Shows how to:
    1. Generate synthetic data with noise and outliers
    2. Initialize StepwiseOutlierDetection orchestrator
    3. Chain multiple detection methods with full parameters
    4. View results at each step
    """
    from diive.pkgs.preprocessing.outlierdetection import StepwiseOutlierDetection

    # Generate synthetic data
    df_noisy = dv.generate_noisy_timeseries(
        start_date='2024-01-01',
        periods=48 * 30,  # ~1 month of half-hourly data
        freq='30min',
        trend_slope=0.01,
        seasonal_strength=9,
        noise_level=2,
        outlier_fraction=0.1  # 10% spikes
    )
    df_noisy.index.name = 'TIMESTAMP_END'

    print(df_noisy.head())

    # Initialize stepwise orchestrator
    detector = StepwiseOutlierDetection(
        dfin=df_noisy,
        col='observed_value',
        site_lat=46.8,
        site_lon=8.6,
        utc_offset=1
    )

    # Chain multiple detectors with full parameter specifications
    print("\n1. Hampel filter (robust spike detection)...")
    detector.flag_outliers_hampel_dtnt_test(
        window_length=7 * 48,
        n_sigma=4.5,
        n_sigma_daytime=None,
        n_sigma_nighttime=None,
        k=1.4826,
        use_differencing=True,
        separate_day_night=True,
        showplot=False,
        verbose=False,
        repeat=True
    )
    print(f"   Found outliers: {(detector.last_flag == 2).sum()}")
    detector.addflag()

    print("\n2. LocalSD (local standard deviation)...")
    detector.flag_outliers_localsd_test(
        n_sd=[3.5, 3.5],  # [daytime_sd, nighttime_sd] when separate_daytime_nighttime=True
        winsize=[24, 24],  # [daytime_winsize, nighttime_winsize] when separate_daytime_nighttime=True
        separate_daytime_nighttime=True,
        showplot=False,
        constant_sd=False,
        verbose=False,
        repeat=True
    )
    print(f"   Found additional outliers: {(detector.last_flag == 2).sum()}")
    detector.addflag()

    print("\n3. Z-score global threshold...")
    detector.flag_outliers_zscore_test(
        thres_zscore=4,
        showplot=False,
        verbose=False,
        plottitle=None,
        repeat=True
    )
    print(f"   Found additional outliers: {(detector.last_flag == 2).sum()}")
    detector.addflag()

    print("\n4. Z-score rolling window (adaptive)...")
    detector.flag_outliers_zscore_rolling_test(
        thres_zscore=3.5,
        showplot=False,
        verbose=False,
        plottitle=None,
        repeat=True,
        winsize=None
    )
    print(f"   Found additional outliers: {(detector.last_flag == 2).sum()}")
    detector.addflag()

    print("\n5. Z-score daytime/nighttime separation...")
    detector.flag_outliers_zscore_dtnt_test(
        thres_zscore=4,
        showplot=False,
        verbose=False,
        repeat=True
    )
    print(f"   Found additional outliers: {(detector.last_flag == 2).sum()}")
    detector.addflag()

    print("\n6. LocalSD increments (abrupt changes)...")
    detector.flag_outliers_increments_zcore_test(
        thres_zscore=3.5,
        showplot=False,
        verbose=False,
        repeat=True
    )
    print(f"   Found additional outliers: {(detector.last_flag == 2).sum()}")
    detector.addflag()

    # View final results
    print(f"\nFinal cleaned series has {detector.series_hires_cleaned.notna().sum()} valid points")
    print(f"All flags:\n{detector.flags.head()}")

    # Visualize original vs cleaned in subplots
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 8))

    # Top plot: Original data with outliers marked
    outliers_mask = detector.series_hires_orig.notna() & detector.series_hires_cleaned.isna()
    ax1.plot(detector.series_hires_orig.index, detector.series_hires_orig.values,
             label='Original data', color='#1f77b4', linewidth=1.2, alpha=0.8)
    ax1.scatter(detector.series_hires_orig[outliers_mask].index,
                detector.series_hires_orig[outliers_mask].values,
                color='#d62728', s=50, label='Detected outliers', zorder=5, alpha=0.8)
    ax1.set_title('Step 1: Original Noisy Data with Detected Outliers', fontsize=12, fontweight='bold')
    ax1.set_ylabel('Value', fontsize=11)
    ax1.legend(loc='upper left', framealpha=0.95)
    ax1.grid(True, alpha=0.3)

    # Bottom plot: Cleaned data
    ax2.plot(detector.series_hires_orig.index, detector.series_hires_orig.values,
             label='Original (for reference)', color='#1f77b4', linewidth=0.8, alpha=0.3, linestyle='--')
    ax2.plot(detector.series_hires_cleaned.index, detector.series_hires_cleaned.values,
             label='Cleaned data (after all filters)', color='#2ca02c', linewidth=1.5, marker='o',
             markersize=3, alpha=0.8)
    ax2.fill_between(detector.series_hires_cleaned.index,
                     detector.series_hires_cleaned.min() - 1,
                     detector.series_hires_cleaned,
                     alpha=0.1, color='#2ca02c')
    ax2.set_title('Step 2: Final Cleaned Series', fontsize=12, fontweight='bold')
    ax2.set_ylabel('Value', fontsize=11)
    ax2.set_xlabel('Timestamp', fontsize=11)
    ax2.legend(loc='upper left', framealpha=0.95)
    ax2.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()


if __name__ == '__main__':
    example_stepwise_outlier_detection_synthetic_data()